# 🌾 An Intelligent System to Track Commodity Price Trends Across Mandis Over Time
### End-to-End Time Series Forecasting (SARIMA), Anomaly Detection, Agmarknet Ingestion & Decision Dashboard

**Author / Project:** Intelligent Mandi Price Tracking & Forecasting System  
**Key Methodologies:** Agmarknet Analytics, STL Seasonal Decomposition, ADF Stationarity Test, Multi-Engine Anomaly Detection (Rolling Z-Score, IQR, Isolation Forest), SARIMA Modeling $(p,d,q) \times (P,D,Q)_s$, Confidence Bands, and Streamlit Dashboard.

## 1. Setup & Dependencies
Install required statistical and visualization packages.

In [ ]:
!pip install -q pandas numpy matplotlib seaborn plotly statsmodels scikit-learn streamlit

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.ensemble import IsolationForest
from sklearn.metrics import mean_absolute_error, mean_squared_error

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print('Libraries successfully imported!')

## 2. Load Agmarknet Dataset
Load cleaned Agmarknet daily price bulletins spanning major Indian mandis.

In [ ]:
# Download or load data from local project repository
data_path = 'data/agmarknet_mandi_prices.csv'

if not os.path.exists(data_path):
    # Sample fallback path or colab upload
    print(f'Please ensure {data_path} is placed in the notebook directory.')

df = pd.read_csv(data_path)
df['Date'] = pd.to_datetime(df['Full_Date'] if 'Full_Date' in df.columns else df['Arrival_Date'], errors='coerce')
df = df.dropna(subset=['Date']).sort_values('Date').reset_index(drop=True)

print(f'Total Agmarknet records loaded: {len(df):,}')
print('Monitored Commodities:', df['Commodity'].unique().tolist())
print('Monitored Mandis:', df['Market'].unique().tolist()[:10])
df.head()

## 3. Exploratory Data Analysis & Cross-Mandi Price Dispersion
Analyze spatial price spreads, highest vs lowest paying mandis, and volatility across crops.

In [ ]:
# Cross-Mandi Price Dispersion by Commodity
plt.figure(figsize=(12, 5))
sns.boxplot(data=df, x='Commodity', y='Modal_Price_Rs_Quintal', palette='Set2')
plt.title('Commodity Modal Price Distributions across Mandis (₹/Quintal)', fontsize=14, fontweight='bold')
plt.ylabel('Modal Price (₹/Quintal)')
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

## 4. Time Series Stationarity Testing (Augmented Dickey-Fuller Test)
Before fitting SARIMA, test for stationarity to determine the required differencing order $d$.

In [ ]:
# Extract continuous daily series for Tomato
commodity = 'Tomato'
crop_df = df[df['Commodity'] == commodity]
daily_ts = crop_df.groupby('Date')['Modal_Price_Rs_Quintal'].mean()

# Reindex to daily continuous and interpolate
full_idx = pd.date_range(daily_ts.index.min(), daily_ts.index.max(), freq='D')
ts_continuous = daily_ts.reindex(full_idx).interpolate(method='linear').bfill().ffill()

# Perform ADF Test
adf_res = adfuller(ts_continuous.values, autolag='AIC')
print(f'=== ADF Stationarity Test for {commodity} ===')
print(f'ADF Statistic: {adf_res[0]:.4f}')
print(f'p-value: {adf_res[1]:.4e}')
for key, val in adf_res[4].items():
    print(f'Critical Value ({key}): {val:.4f}')

if adf_res[1] < 0.05:
    print('=> The series is Stationary at 5% significance level (d=0).')
else:
    print('=> The series is Non-Stationary (p >= 0.05). First-order differencing (d=1) required for SARIMA.')

## 5. STL & Seasonal Decomposition
Deconstruct the price series into Trend ($T_t$), Seasonality ($S_t$, period=7 days for weekly mandi cycles), and Residual Noise ($R_t$).

In [ ]:
decomp = seasonal_decompose(ts_continuous, model='additive', period=7, extrapolate_trend='freq')

fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)
decomp.observed.plot(ax=axes[0], color='#0f172a', title='Observed Modal Price (₹/Quintal)')
decomp.trend.plot(ax=axes[1], color='#059669', title='Long-Term Trend Component')
decomp.seasonal.plot(ax=axes[2], color='#2563eb', title='7-Day Seasonal Repeat Cycle')
decomp.resid.plot(ax=axes[3], color='#dc2626', style='.', title='Irregular Residual Variations')
plt.tight_layout()
plt.show()

## 6. Multi-Engine Anomaly Detection
Identify price spikes, supply deficit shocks, and unseasonal crashes using:
1. **Dynamic Rolling Z-Score** ($|Z| \ge 2.5\sigma$)
2. **Interquartile Range (IQR)**
3. **Isolation Forest** (Unsupervised Tree Outliers)

In [ ]:
# 1. Rolling Z-Score
roll_mean = ts_continuous.rolling(window=14, min_periods=3).mean()
roll_std = ts_continuous.rolling(window=14, min_periods=3).std().fillna(1)
z_scores = (ts_continuous - roll_mean) / roll_std
anom_z = np.abs(z_scores) >= 2.5

# 2. Isolation Forest
features = pd.DataFrame({
    'return': ts_continuous.pct_change().fillna(0),
    'roll_std_7': ts_continuous.rolling(7, min_periods=1).std().fillna(0)
})
iso = IsolationForest(contamination=0.03, random_state=42)
anom_iso = (iso.fit_predict(features) == -1)

plt.figure(figsize=(14, 5))
plt.plot(ts_continuous.index, ts_continuous.values, label='Modal Price', color='#64748b', lw=1.5)
plt.scatter(ts_continuous.index[anom_z], ts_continuous.values[anom_z], color='#dc2626', label='Z-Score Anomalies (|Z|>=2.5σ)', s=50, marker='x')
plt.title(f'{commodity} Price Trajectory with Flagged Price Shock Anomalies', fontsize=13, fontweight='bold')
plt.ylabel('Price (₹/Quintal)')
plt.legend()
plt.tight_layout()
plt.show()

print(f'Total Z-Score Anomalies: {anom_z.sum()}')
print(f'Total Isolation Forest Anomalies: {anom_iso.sum()}')

## 7. SARIMA Forecasting Formulation & Backtesting
Fit a Seasonal AutoRegressive Integrated Moving Average $SARIMA(p, d, q) \times (P, D, Q)_s$ model.
Evaluate on holdout test set using **MAE**, **RMSE**, and **MAPE**.

In [ ]:
# Split into Train and Test (Holdout 28 days)
test_days = 28
train = ts_continuous.iloc[:-test_days]
test = ts_continuous.iloc[-test_days:]

# Fit SARIMA(1, 1, 1) x (1, 0, 1, 7)
order = (1, 1, 1)
seasonal_order = (1, 0, 1, 7)

print(f'Training SARIMA {order} x {seasonal_order} on {len(train)} observations...')
model = SARIMAX(train, order=order, seasonal_order=seasonal_order, enforce_stationarity=False, enforce_invertibility=False)
sarima_fit = model.fit(disp=False)

# Forecast holdout test
pred_test = sarima_fit.get_forecast(steps=test_days)
test_forecast = pred_test.predicted_mean
ci = pred_test.conf_int(alpha=0.05)

# Metrics
mae = mean_absolute_error(test, test_forecast)
rmse = np.sqrt(mean_squared_error(test, test_forecast))
mape = np.mean(np.abs((test - test_forecast) / test)) * 100

print(f'=== Holdout Test Performance (28 Days) ===')
print(f'MAE:  ₹{mae:.2f}')
print(f'RMSE: ₹{rmse:.2f}')
print(f'MAPE: {mape:.2f}%')
print(f'AIC:  {sarima_fit.aic:.1f}')

## 8. Multi-Step Forward Out-of-Sample Forecasting
Train SARIMA on full historical series and project forward 30 days with a 95% Confidence Interval cone.

In [ ]:
# Re-train on full historical data
full_model = SARIMAX(ts_continuous, order=order, seasonal_order=seasonal_order, enforce_stationarity=False, enforce_invertibility=False)
full_fit = full_model.fit(disp=False)

horizon = 30
forecast_res = full_fit.get_forecast(steps=horizon)
forecast_mean = forecast_res.predicted_mean
ci_95 = forecast_res.conf_int(alpha=0.05)

# Plot Forward Forecast with 95% Confidence Interval
plt.figure(figsize=(14, 6))
plt.plot(ts_continuous.index[-90:], ts_continuous.values[-90:], label='Historical Price (Recent 90D)', color='#059669', lw=2)
plt.plot(forecast_mean.index, forecast_mean.values, label=f'SARIMA Forecast ({horizon}D Ahead)', color='#d97706', lw=2.5, ls='--')
plt.fill_between(forecast_mean.index, ci_95.iloc[:, 0], ci_95.iloc[:, 1], color='#f59e0b', alpha=0.2, label='95% Confidence Band')

plt.title(f'SARIMA Forward Price Projection for {commodity} with 95% Confidence Cone', fontsize=14, fontweight='bold')
plt.ylabel('Modal Price (₹/Quintal)')
plt.legend(loc='upper left')
plt.tight_layout()
plt.show()

## 9. Launch Interactive Streamlit Dashboard
Run the full web dashboard locally or expose it via Colab tunneling.

In [ ]:
# To run the interactive dashboard in Colab:
# !streamlit run app.py & npx localtunnel --port 8501